# Shared-signal Food PoC

Attach exactly two Kaggle datasets: the opaque `*.tar.gz.b64` corpus payload and the `*.tar.gz` selected Foundation checkpoint package produced by the pretraining notebook.


In [ ]:
from pathlib import Path
import subprocess, sys
repo = Path('/kaggle/working/repo')
if not repo.exists():
    subprocess.run(['git', 'clone', '--branch', 'feat/shared-signal-poc', 'https://github.com/huongiang171/green-sm-foundation-pretraining.git', str(repo)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo)], check=True)
print('Repo:', repo, flush=True)


In [ ]:
import hashlib, json, sys, tarfile
from pathlib import Path, PurePosixPath
import yaml
sys.path.insert(0, str(repo))
from scripts.package_shared_signal_poc import extract_shared_signal_poc

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def extract_checkpoint(archive_path, destination):
    with tarfile.open(archive_path, 'r:gz') as archive:
        for member in archive.getmembers():
            parts = PurePosixPath(member.name).parts
            assert not PurePosixPath(member.name).is_absolute() and '..' not in parts
            if member.isdir():
                continue
            assert member.isfile() and parts
            output = destination.joinpath(*parts)
            output.parent.mkdir(parents=True, exist_ok=True)
            output.write_bytes(archive.extractfile(member).read())

inputs = Path('/kaggle/input')
payloads = list(inputs.rglob('*.tar.gz.b64'))
checkpoints = list(inputs.rglob('*.tar.gz'))
assert len(payloads) == 1 and len(checkpoints) == 1, (payloads, checkpoints)
corpus_root = Path('/tmp/green_sm_shared_signal_poc_v1')
checkpoint_root = Path('/tmp/green_sm_shared_signal_poc_checkpoint')
extract_shared_signal_poc(payloads[0], corpus_root)
extract_checkpoint(checkpoints[0], checkpoint_root)
checkpoint_manifests = list(checkpoint_root.rglob('checkpoint_manifest.json'))
assert len(checkpoint_manifests) == 1, checkpoint_manifests
checkpoint_root = checkpoint_manifests[0].parent
selected = json.loads(checkpoint_manifests[0].read_text())
linked_root = corpus_root / 'green_sm_shared_signal_poc_linked_v1'
food_root = linked_root / 'food_opportunities'
foundation_root = linked_root / 'foundation'
config = yaml.safe_load((repo / 'configs/food_recommendation_shared_signal_poc.yaml').read_text())
config['data'].update({'opportunities_root': str(food_root), 'manifest_path': str(food_root / 'manifest.json'), 'foundation_root': str(foundation_root), 'foundation_manifest_path': str(foundation_root / 'manifest.json'), 'vocabulary_path': str(checkpoint_root / 'vocabulary.json'), 'expected_food_manifest_sha256': sha256(food_root / 'manifest.json'), 'expected_foundation_manifest_sha256': sha256(foundation_root / 'manifest.json')})
config['checkpoint'].update({'package_root': str(checkpoint_root), 'expected_selected_checkpoint_sha256': selected['checkpoint_sha256']})
run_dir = Path('/kaggle/working/artifacts/food_shared_signal_poc/run')
config['ranker']['run_dir'] = str(run_dir)
config['training']['artifact_dir'] = str(run_dir)
runtime = Path('/kaggle/working/food_shared_signal_runtime.yaml')
runtime.write_text(yaml.safe_dump(config, sort_keys=False))
print('Corpus:', linked_root, 'Checkpoint kind:', selected['selected_kind'], 'Runtime:', runtime, flush=True)


In [ ]:
%cd /kaggle/working/repo
!python -u -m src.downstream.food_recommendation --config /kaggle/working/food_shared_signal_runtime.yaml --mode validate
!python -u -m src.downstream.food_recommendation --config /kaggle/working/food_shared_signal_runtime.yaml --mode train


In [ ]:
import hashlib, yaml
run_manifest = Path('/kaggle/working/artifacts/food_shared_signal_poc/run/run_manifest.json')
config = yaml.safe_load(Path('/kaggle/working/food_shared_signal_runtime.yaml').read_text())
config['ranker']['expected_run_manifest_sha256'] = sha256(run_manifest)
Path('/kaggle/working/food_shared_signal_runtime.yaml').write_text(yaml.safe_dump(config, sort_keys=False))
print('Food run manifest SHA:', config['ranker']['expected_run_manifest_sha256'], flush=True)
!python -u -m src.downstream.food_recommendation --config /kaggle/working/food_shared_signal_runtime.yaml --mode predict --split validation
!python -u -m src.downstream.food_recommendation --config /kaggle/working/food_shared_signal_runtime.yaml --mode predict --split test


In [ ]:
!find /kaggle/working/artifacts/food_shared_signal_poc -maxdepth 5 -type f | sort
!cd /kaggle/working/artifacts && tar -czf /kaggle/working/food_shared_signal_poc_run.tar.gz food_shared_signal_poc
!ls -lh /kaggle/working/food_shared_signal_poc_run.tar.gz
